<a href="https://colab.research.google.com/github/engmodu/AIFEL_quest_eng/blob/main/NLP/NLP05/HuggingFacePrj_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#STEP 1. NSMC 데이터 분석 및 Huggingface dataset 구성
데이터셋은 깃허브에서 다운받거나, Huggingface datasets에서 가져올 수 있습니다. 앞에서 배운 방법들을 활용해봅시다!
#STEP 2. klue/bert-base model 및 tokenizer 불러오기
#STEP 3. 위에서 불러온 tokenizer으로 데이터셋을 전처리하고, model 학습 진행해 보기
#STEP 4. Fine-tuning을 통하여 모델 성능(accuarcy) 향상시키기
데이터 전처리, TrainingArguments 등을 조정하여 모델의 정확도를 90% 이상으로 끌어올려봅시다.
#STEP 5. Bucketing을 적용하여 학습시키고, STEP 4의 결과와의 비교
아래 링크를 바탕으로 bucketing과 dynamic padding이 무엇인지 알아보고, 이들을 적용하여 model을 학습시킵니다.

#전체 그림
1.NSMC 데이터(document, label)

2.Hugging Face Dataset

3.tokenizer(input_ids, attention_mask, label)

4.klue/bert-base

5.Trainer 학습

6.accuracy 평가

7.bucketing + dynamic padding 적용 후 비교

In [ ]:
#import tensorflow
import numpy
import transformers
import datasets

print(tensorflow.__version__)
print(numpy.__version__)
print(transformers.__version__)
print(datasets.__version__)


from google.colab import drive
drive.mount('/content/drive')

import json

home_path = "/content/drive/MyDrive/Colab Notebooks/"

#STEP 0. 설치

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

# STEP 1. NSMC 데이터 불러오기

In [ ]:
from datasets import load_dataset

data_files = {
    "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
    "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
}

dataset = load_dataset(
    "csv",
    data_files=data_files,
    delimiter="\t"
)

dataset

In [ ]:
from datasets import load_dataset

data_files = {
    "train": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
    "test": "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
}

dataset = load_dataset(
    "csv",
    data_files=data_files,
    delimiter="\t"
)

dataset

# STEP 1-1. 데이터 확인

In [ ]:
dataset["train"][0]

In [ ]:
dataset["train"].features

In [ ]:
print(dataset["train"].column_names)
print(dataset["test"].column_names)

# STEP 1-2. 결측치 제거

In [ ]:
dataset = dataset.filter(lambda x: x["document"] is not None)

In [ ]:
dataset["train"][0]

#STEP 1-3. 빠른 실험용 샘플링

처음부터 전체 데이터로 돌리면 오래 걸릴 수 있으니 먼저 작은 데이터로 확인한다.

In [ ]:
#small_train = dataset["train"].shuffle(seed=42).select(range(10000))
#small_val = dataset["test"].shuffle(seed=42).select(range(2000))

In [ ]:
small_train = dataset["train"]
small_val = dataset["test"]

#STEP 2. klue/bert-base 모델과 토크나이저 불러오기

klue/bert-base는 한국어 KLUE 벤치마크 개발 맥락에서 공개된 한국어 BERT base 모델이다.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)
#여기서 classifier.weight, classifier.bias가 새로 초기화됐다는 경고가 나올 수 있다.
#정상이다. 기존 BERT 위에 NSMC 긍정/부정 분류용 머리를 새로 붙이는 과정이다.

#STEP 3. 토크나이징 전처리
고정 패딩 방식

모든 문장을 max_length=128 길이로 맞춘다.

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["document"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

In [ ]:
tokenized_train = small_train.map(tokenize_function, batched=True)
tokenized_val = small_val.map(tokenize_function, batched=True)

In [ ]:
tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

#STEP 3-1. 평가 함수 만들기

In [ ]:
import evaluate
import numpy as np

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

#STEP 3-2. 기본 학습

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./nsmc-klue-bert-basic",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none"
)
#구버전 transformers에서 eval_strategy 오류가 나면 이것으로 바꿔라.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

#STEP 4. Fine-tuning으로 정확도 올리기

In [ ]:
train_dataset = dataset["train"]
val_dataset = dataset["test"]

In [ ]:
tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["id", "document"])
tokenized_val = tokenized_val.remove_columns(["id", "document"])

tokenized_train.set_format("torch")
tokenized_val.set_format("torch")

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./nsmc-klue-bert-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    report_to="none"
)

#GPU에서 fp16=True 오류가 나면 fp16=False로 바꿔라.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
finetune_result = trainer.evaluate()
finetune_result

STEP 5. Bucketing + Dynamic Padding 적용
개념 그림

기본 padding:

문장 A 길이 10  →  128까지 padding
문장 B 길이 20  →  128까지 padding
문장 C 길이 70  →  128까지 padding

낭비가 많다.

Dynamic padding:

한 배치 안에서 가장 긴 문장 길이까지만 padding

Bucketing:

길이 비슷한 문장끼리 묶음
짧은 문장끼리 batch
긴 문장끼리 batch

효과:

padding 감소
↓
연산량 감소
↓
학습 시간 단축

STEP 5-1. Dynamic padding용 전처리

이번에는 padding="max_length"를 제거한다.

In [ ]:
def tokenize_no_padding(batch):
    return tokenizer(
        batch["document"],
        truncation=True,
        max_length=128
    )

In [ ]:
bucket_train = dataset["train"].map(tokenize_no_padding, batched=True)
bucket_val = dataset["test"].map(tokenize_no_padding, batched=True)

bucket_train = bucket_train.remove_columns(["id", "document"])
bucket_val = bucket_val.remove_columns(["id", "document"])

bucket_train.set_format("torch")
bucket_val.set_format("torch")

STEP 5-2. DataCollatorWithPadding 사용

In [ ]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)
#DataCollatorWithPadding은 배치가 만들어질 때 그 배치 안에서 필요한 만큼만 padding을 붙인다.

STEP 5-3. Bucketing 학습 설정

In [ ]:
group_by_length=True

In [ ]:
bucket_model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

In [ ]:
bucket_training_args = TrainingArguments(
    output_dir="./nsmc-klue-bert-bucketing",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=500,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=True,
    group_by_length=True,
    report_to="none"
)

In [ ]:
bucket_trainer = Trainer(
    model=bucket_model,
    args=bucket_training_args,
    train_dataset=bucket_train,
    eval_dataset=bucket_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
bucket_trainer.train()

In [ ]:
bucket_result = bucket_trainer.evaluate()
bucket_result

STEP 6. 결과 비교표 만들기

In [ ]:
import pandas as pd

compare_df = pd.DataFrame([
    {
        "method": "fine_tuning_fixed_padding",
        "accuracy": finetune_result.get("eval_accuracy"),
        "loss": finetune_result.get("eval_loss"),
    },
    {
        "method": "bucketing_dynamic_padding",
        "accuracy": bucket_result.get("eval_accuracy"),
        "loss": bucket_result.get("eval_loss"),
    },
])

compare_df

In [ ]:
import matplotlib.pyplot as plt

methods = compare_df["method"]
accuracy = compare_df["accuracy"]
loss = compare_df["loss"]

# Accuracy 그래프
plt.figure(figsize=(6,4))
plt.bar(methods, accuracy)
plt.ylabel("Accuracy")
plt.title("Accuracy Comparison")
plt.ylim(0.90, 0.91)
plt.xticks(rotation=10)

for i, v in enumerate(accuracy):
    plt.text(i, v + 0.0001, f"{v:.4f}", ha='center')

plt.show()

최종 해석 기준
accuracy 측면:
fine-tuning 설정이 좋아지면 정확도 상승 가능

훈련 시간 측면:
bucketing + dynamic padding은 padding 토큰을 줄여서 보통 더 효율적

성능 측면:
bucketing 자체가 정확도를 직접 올리는 기술은 아님
하지만 batch 구성과 padding 효율이 좋아져 학습이 안정될 수 있음

정리하면:

정확도 향상 핵심:
learning_rate, epoch, batch_size, max_length, warmup_ratio

속도 향상 핵심:
DataCollatorWithPadding + group_by_length=True